In [25]:
import os
import scanpy as sc
import celltypist
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import tarfile
import urllib.request
import shutil
import glob
import ssl
import gzip
import re  # For robust T2G parsing

# Suppress warnings
warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

print("Environment setup complete.")

Environment setup complete.


In [ ]:
# --- 1. Setup Data Directory ---
os.makedirs("data", exist_ok=True)

# --- 2. Download Whitelist ---
url = "https://github.com/f0t1h/3M-february-2018/raw/refs/heads/master/3M-february-2018.txt.gz"
whitelist_path = "data/whitelist.txt.gz"

ssl_context = ssl.create_default_context()
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

if not os.path.exists("data/whitelist.txt"):
    print("Downloading whitelist...")
    try:
        with urllib.request.urlopen(url, context=ssl_context) as response, open(whitelist_path, 'wb') as out_file:
            shutil.copyfileobj(response, out_file)
        os.system(f"gunzip -f {whitelist_path}")
        print("Whitelist ready.")
    except Exception as e:
        print(f"Whitelist download failed: {e}")

# --- 3. Extract Input Data ---
tar_filename = "toy_read_ref_set.tar.gz"
if os.path.exists(tar_filename):
    print(f"Extracting {tar_filename}...")
    try:
        with tarfile.open(tar_filename, "r:*") as tar:
            tar.extractall()
        print("Extraction complete.")
    except Exception as e:
        print(f"Extraction failed: {e}")
else:
    print(f"WARNING: {tar_filename} not found in current directory.")

# --- 4. Find and Rename Files ---
target_genome = "genome.fa"
target_gtf = "genes.gtf"
target_r1 = "r1.fq.gz"
target_r2 = "r2.fq.gz"

def safe_rename(src, dst):
    if os.path.abspath(src) == os.path.abspath(dst):
        return
    print(f"Renaming {src} -> {dst}")
    shutil.move(src, dst)

def find_file(extensions):
    candidates = []
    for root, _, files in os.walk("."):
        if "data/" in root:
            continue
        for f in files:
            if f.lower().endswith(extensions) and not f.startswith("._"):
                candidates.append(os.path.join(root, f))
    return candidates

# Find Genome
genomes = find_file(('.fa', '.fasta'))
if genomes:
    safe_rename(max(genomes, key=os.path.getsize), target_genome)
else:
    print("WARNING: No genome (.fa/.fasta) file found.")

# Find GTF
gtfs = find_file(('.gtf',))
if gtfs:
    safe_rename(max(gtfs, key=os.path.getsize), target_gtf)
else:
    print("WARNING: No GTF (.gtf) file found.")

# Find FASTQs (Compressed or Uncompressed)
fastqs = [f for f in find_file(('.fastq.gz', '.fq.gz', '.fastq', '.fq')) if "whitelist" not in f]
fastqs = sorted(fastqs)

def compress_and_move(src, dest):
    if src.endswith('.gz'):
        safe_rename(src, dest)
    else:
        print(f"Compressing {src} to {dest}...")
        with open(src, 'rb') as f_in:
            with gzip.open(dest, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)

if len(fastqs) >= 2:
    compress_and_move(fastqs[0], target_r1)
    compress_and_move(fastqs[1], target_r2)
else:
    print("WARNING: Could not find 2 FASTQ files. Current dir:")
    os.system("ls -R")

# --- 5. Generate Transcriptome ---
if not os.path.exists("transcripts.fa"):
    print("Installing gffread...")
    os.system("mamba install -y -c bioconda gffread")
    print("Generating transcripts.fa...")
    os.system(f"gffread -w transcripts.fa -g {target_genome} {target_gtf}")
else:
    print("transcripts.fa already exists, skipping generation.")

In [ ]:
print("Building transcript-to-gene mapping (t2g)...")

gtf_path = "genes.gtf"
t2g_path = "data/t2g.tsv"
os.makedirs("data", exist_ok=True)

n_lines = 0

with open(gtf_path) as gtf, open(t2g_path, "w") as out:
    for line in gtf:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9:
            continue

        feature = fields[2]
        # only keep transcript records
        if feature != "transcript":
            continue

        attrs_field = fields[8]
        # parse attributes like: key "value";
        attrs = {
            m.group(1): m.group(2)
            for m in re.finditer(r'(\S+)\s+"([^"]+)"', attrs_field)
        }

        tid = attrs.get("transcript_id")
        gid = attrs.get("gene_id")
        gname = attrs.get("gene_name", gid if gid else "")

        if not tid or not gid:
            continue

        out.write(f"{tid}\t{gid}\t{gname}\n")
        n_lines += 1

print(f"Wrote {n_lines} transcript→gene mappings to {t2g_path}")

In [ ]:
%%bash
set -e

TRANSCRIPTS="transcripts.fa"
INDEX_DIR="data/salmon_index"
R1="r1.fq.gz"
R2="r2.fq.gz"
MAP_OUT="data/alevin_out"

mkdir -p "$(dirname "$INDEX_DIR")"

echo "Building salmon index (if needed)..."
if [ ! -d "$INDEX_DIR" ] || [ -z "$(ls -A "$INDEX_DIR" 2>/dev/null)" ]; then
    salmon index -t "$TRANSCRIPTS" -i "$INDEX_DIR"
else
    echo "Salmon index already exists at $INDEX_DIR"
fi

echo "Running salmon alevin..."
rm -rf "$MAP_OUT"
salmon alevin \
  -l A \
  -i "$INDEX_DIR" \
  -1 "$R1" \
  -2 "$R2" \
  --chromium \
  --whitelist data/whitelist.txt \
  --rad \
  -p 4 \
  -o "$MAP_OUT"

echo "Salmon alevin finished, RAD output in $MAP_OUT"

In [ ]:
%%bash
set -e
MAP_OUT="data/alevin_out"
QUANT_OUT="data/fry_quant"
WHITELIST="data/whitelist.txt"
T2G="data/t2g.tsv"

# Clean up
rm -rf "$QUANT_OUT"

# 1. Generate Permit List
echo "Generating permit list..."
if ! alevin-fry generate-permit-list \
    -d fw \
    -i "$MAP_OUT" \
    -o "$QUANT_OUT" \
    -u "$WHITELIST" > fry_permit.log 2>&1; then
    echo "ERROR: generate-permit-list failed! Log:"
    cat fry_permit.log
    exit 1
fi

# 2. Collate
echo "Collating..."
if ! alevin-fry collate \
    -i "$QUANT_OUT" \
    -r "$MAP_OUT" \
    -t 2 > fry_collate.log 2>&1; then
    echo "ERROR: collate failed! Log:"
    cat fry_collate.log
    exit 1
fi

# 3. Quantify
echo "Quantifying..."
if ! alevin-fry quant \
    -i "$QUANT_OUT" \
    -o "$QUANT_OUT/res" \
    -t 2 \
    -r cr-like \
    -m "$T2G" \
    --use-mtx > fry_quant.log 2>&1; then
    echo "ERROR: quant failed! Log:"
    cat fry_quant.log
    exit 1
fi

# 4. Verify Output
TARGET_MTX="$QUANT_OUT/res/alevin/quants_mat.mtx"
if [ -f "$TARGET_MTX" ]; then
    # Make a gzipped copy but keep the original .mtx
    gzip -c "$TARGET_MTX" > "$TARGET_MTX.gz"
    echo "Success: quants_mat.mtx and quants_mat.mtx.gz created."
else
    echo "ERROR: quants_mat.mtx NOT found. Listing output dir:"
    ls -R "$QUANT_OUT"
    exit 1
fi

In [ ]:
print("Loading count matrix...")

from scipy.io import mmread
import pandas as pd

mtx_path = "data/fry_quant/res/alevin/quants_mat.mtx"
rows_path = "data/fry_quant/res/alevin/quants_mat_rows.txt"
cols_path = "data/fry_quant/res/alevin/quants_mat_cols.txt"

# Read matrix (genes x cells)
X = mmread(mtx_path).tocsr()

# Read gene and cell names
genes = pd.read_csv(rows_path, header=None)[0].astype(str)
cells = pd.read_csv(cols_path, header=None)[0].astype(str)

# Alevin-fry: rows = genes, cols = cells → Scanpy expects cells x genes
X = X.T.tocsr()

adata = sc.AnnData(X=X)
adata.var_names = genes.values
adata.obs_names = cells.values

print(adata)

# QC & Filter
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.pct_counts_mt < 5, :]

# Normalize & Cluster
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)
sc.tl.leiden(adata)

sc.pl.umap(adata, color=["leiden"], title="Leiden Clustering", show=True)

In [ ]:
model = celltypist.models.Model.load(model="Immune_All_Low.pkl")
predictions = celltypist.annotate(adata, model="Immune_All_Low.pkl", majority_voting=True)
adata.obs["cell_type"] = predictions.predicted_labels["predicted_labels"]
adata.obs["conf_score"] = predictions.predicted_labels["conf_score"]

sc.pl.umap(
    adata,
    color=["cell_type"],
    title="CellTypist Annotation",
    legend_loc="on data"
)

print(adata.obs["cell_type"].value_counts())